# Chains (LangChain v1.2)

**LCEL(LangChain Expression Language)**을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**
1. **PromptTemplate**  
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합  
   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**  
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현  
   ```python
   model = ChatOpenAI(model="gpt-4o")
   ```

3. **Memory**  
   - `RunnableWithMessageHistory`로 통합 관리 (또는 LangGraph Persistence 사용)
   ```python
   chain_with_memory = RunnableWithMessageHistory(
       base_chain,
       get_session_history
   )
   ```

4. **Output Parsers**  
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동  
   ```python
   output_parser = JsonOutputParser()
   ```

5. **Tools**  
   - `@tool` 데코레이터로 생성 후 `RunnableLambda`로 변환  
   ```python
   @tool
   def search(query: str) -> str: ...
   ```

**체인 유형별 구현**


1. Simple Chain  

    ```python
    chain = prompt | model | output_parser
    response = chain.invoke({"input": "..."})
    ```

2. Sequential Chain  

    ```python
    chain = (
        {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
        | prompt2
        | model2
    )
    ```

3. Conditional Chain
    - `RunnableBranch` 사용

    ```python
    branch = RunnableBranch(
        (lambda x: x["topic"] == "math", math_chain),
        (lambda x: x["topic"] == "history", history_chain),
        default_chain
    )
    ```

4. Memory Chain  

    ```python
    memory_chain = RunnableWithMessageHistory(
        core_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    ```


**🚨 v1.2 주요 변경점**

- **Legacy Chain 클래스 완전 폐기**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable` (LCEL)로 통합
- **에이전트 통합**: `create_agent` (LangGraph 기반)가 표준

In [1]:
%pip install -Uqqq langchain langchain-openai langchain-community

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template('{city}의 특산물은?')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser
print(chain.invoke('강원도'))
# print(chain.invoke(input = {'city': '강원도', ...}))

강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 평창·강릉·홍천 등, 강원도를 대표하는 농산물  
- **옥수수**: 홍천 찰옥수수, 강릉·정선 옥수수  
- **황태**: 인제 용대리 황태가 유명  
- **오징어**: 동해안의 대표 수산물  
- **한우**: 횡성한우, 평창한우  
- **곤드레**: 정선 곤드레나물  
- **메밀**: 평창 봉평 메밀, 메밀전병·막국수  
- **더덕**: 횡성·정선·홍천 등에서 생산  
- **송이버섯**: 양양 송이가 대표적  
- **잣**: 홍천 잣  
- **초당두부**: 강릉 초당 지역의 전통 음식  
- **닭갈비·막국수**: 춘천을 대표하는 향토 음식  

지역별로는 **평창 감자·메밀, 횡성 한우, 인제 황태, 양양 송이, 홍천 잣, 정선 곤드레**가 특히 유명합니다.


In [4]:
prompt1 = PromptTemplate.from_template('다음 내용 한글로 번역. {eng_text}')
prompt2 = PromptTemplate.from_template('다음 내용 요약. {kor_text}')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

chain1 = prompt1 | llm
eng_text = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""
print(chain1.invoke(eng_text))

chain2 = prompt2 | llm | output_parser
kor_text="""
LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보를 갖고 있지 않다는 점입니다. 이를 해결하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.
이를 위해서는 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF, 이메일부터 웹사이트, 유튜브 영상에 이르기까지 다양한 유형의 문서를 위한 여러 종류의 로더를 제공합니다.
"""
print(chain2.invoke(kor_text))

content='LLM의 한 가지 한계는 맥락 정보가 부족하다는 점입니다. 예를 들어 특정 문서나 이메일에 접근할 수 없습니다. 이러한 한계를 극복하려면 LLM이 외부의 특정 데이터에 접근할 수 있도록 해야 합니다.\n\n이를 위해 먼저 문서 로더(document loader)를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF와 이메일부터 웹사이트 및 YouTube 동영상에 이르기까지 다양한 유형의 문서를 지원하는 여러 로더를 제공합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 96, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EH2UbgE2Pn9Exf15Bpqjiy4mIEcyC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a03d00-e442-7323-acb8-69c8d0d5dc85-0' tool_calls=[] invalid_tool_calls=[] usage_metadata=

In [5]:
chain = chain1 | chain2
print(chain.invoke({'eng_text': eng_text}))

LLM은 외부 문서나 이메일 등 특정 맥락 정보에 직접 접근하기 어렵다는 한계가 있습니다. 이를 해결하기 위해 문서 로더를 사용해 외부 데이터를 불러오며, LangChain은 PDF, 이메일, 웹사이트, YouTube 동영상 등 다양한 형식의 로더를 제공합니다.


In [ ]:
from langchain_core.runnables import RunnableBranch

llm = init_chat_model('gpt-5.6-luna')

math_prompt = PromptTemplate.from_template('다음 문제 해결. 단계적 풀이를 LaTex 수식과 함께 작성. {question}')
math_chain = math_prompt | llm | output_parser

default_prompt = PromptTemplate.from_template('당신은 공감능력 좋은 챗봇. 다음 질문에 답변. {question}')
default_chain = default_prompt | llm | output_parser

def is_math_question(input_dict: dict) -> bool:
    question: str = input_dict.get('question', '')
    return '계산' in question or 'calc' in question

branch_chain = RunnableBranch(
    (is_math_question, math_chain),
    default_chain
)

print(branch_chain.invoke({'question': '125 * 3 + 50 계산'}))

'단계적으로 계산하면 다음과 같습니다.\n\n\\[\n125 \\times 3 = 375\n\\]\n\n따라서,\n\n\\[\n125 \\times 3 + 50 = 375 + 50 = 425\n\\]\n\n\\[\n\\boxed{425}\n\\]'

In [8]:
print(branch_chain.invoke({'question': '오늘 우울한데 빵 or 밥'}))

오늘처럼 우울한 날엔 **먹기 편하고 속이 편한 쪽**이 최고예요.

- 든든하게 오래 버티고 싶으면: **밥 + 계란/김/국**
- 입맛 없고 간단히 먹고 싶으면: **빵 + 우유/요거트/계란**

굳이 고르라면 오늘은 **따뜻한 밥** 추천할게요. 하지만 빵이 더 당긴다면 빵 먹어도 충분히 괜찮아요. 일단 뭐라도 조금 먹는 게 중요해요.


In [9]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from pydantic import BaseModel, Field
from typing import List

class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    messages: List[BaseMessage] = Field(default_factory=list)

    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)

    def clear(self) -> None:
        self.messages = []

store = {}

def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()
    return store[session_id]

history1 = get_by_session_id('1')
history1.add_messages([AIMessage(content='반갑습니다. Capybara님')])
history1.add_messages([HumanMessage(content='그래 반갑다')])
print(f'{history1 = }')

history2 = get_by_session_id('2')
print(f'{history2 = }')

history1 = InMemoryHistory(messages=[AIMessage(content='반갑습니다. Capybara님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래 반갑다', additional_kwargs={}, response_metadata={})])
history2 = InMemoryHistory(messages=[])


In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야 전문가 챗봇'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,
    input_messages_key='question',
    history_messages_key='history'
)

chain_with_history.invoke({
    'domain': 'math',
    'question': '민수는 강아지 3마리 키운다'
    }, config={
        'configurable': {
            'session_id': '100'
        }
    }
)

AIMessage(content='민수는 강아지 3마리를 키우고 있군요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 33, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 38, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EH3Vp8q0qThtuoTTcwbGbH5CIGKwz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a03d3c-b648-7b13-9143-42576efe56fd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 64, 'total_tokens': 97, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 38}})

In [12]:
chain_with_history.invoke({
    'domain': 'math',
    'question': '소라는 고양이 4마리 키운다'
    }, config={
        'configurable': {
            'session_id': '100'
        }
    }
)

AIMessage(content='소라는 고양이 4마리를 키우고 있군요. 민수의 강아지 3마리와 합하면 총 7마리의 반려동물입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 72, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 56, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHI5PKv1RQQHbJKECxQCIQoyDZTvA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04093-9271-7142-bbe6-f6b5ab5bc83d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 106, 'total_tokens': 178, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, '

In [13]:
store

{'1': InMemoryHistory(messages=[AIMessage(content='반갑습니다. Capybara님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래 반갑다', additional_kwargs={}, response_metadata={})]),
 '2': InMemoryHistory(messages=[]),
 '100': InMemoryHistory(messages=[HumanMessage(content='민수는 강아지 3마리 키운다', additional_kwargs={}, response_metadata={}), AIMessage(content='민수는 강아지 3마리를 키우고 있군요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 33, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 38, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EH3Vp8q0qThtuoTTcwbGbH5CIGKwz', 'se

In [14]:
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}

def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야 전문가 챗봇'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}')
])


llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm | output_parser

chain_with_history.invoke({
    'domain': '심리상담',
    'question': '요즘 더워져서 화난다. 원래 성격 좋은데 왜 이러지'
    }, config={
        'configurable': {
            'session_id': '200'
        }
    })

AIMessage(content='더위 때문에 예민해지는 건 꽤 자연스러운 반응이에요. 성격이 나빠진 게 아니라, 몸이 열을 식히느라 에너지를 많이 쓰고 잠의 질·수분 균형·집중력이 떨어지면서 짜증과 분노를 조절하기 어려워질 수 있습니다. 특히 더운 날씨에 피로, 탈수, 수면 부족이 겹치면 더 심해져요.\n\n도움이 될 만한 방법은:\n\n- 물을 조금씩 자주 마시고, 땀을 많이 흘렸다면 전해질도 보충하기  \n- 실내 온도와 습도 낮추기, 외출·운동은 비교적 선선한 시간에 하기  \n- 더울 때 중요한 대화나 결정은 잠시 미루고 “지금 더워서 예민해졌으니 10분 쉬자”고 말하기  \n- 찬물 세수, 목·겨드랑이·손목 식히기, 헐렁한 옷 입기  \n- 수면을 우선하고 카페인·술은 줄이기  \n- 화가 올라올 때 바로 반응하기보다 천천히 숨을 내쉬며 몸부터 식히기\n\n다만 시원한 환경에서도 계속 심하게 화가 나거나, 우울·불안·불면이 함께 이어지거나, 두근거림·어지럼·심한 두통·메스꺼움이 있으면 단순한 더위가 아닐 수 있으니 진료를 받아보세요. 최근 잠이나 스트레스 상태도 함께 돌아보면 원인을 찾는 데 도움이 됩니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 402, 'prompt_tokens': 43, 'total_tokens': 445, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 34, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_t

In [15]:
chain_with_history.invoke({
    'domain': '심리상담',
    'question': '날씨 좋게 바꿔'
    }, config={
        'configurable': {
            'session_id': '200'
        }
    })

AIMessage(content='저도 당장 바꿔드리고 싶네요 😅  \n제가 날씨를 직접 바꿀 수는 없지만, **오늘만큼은 덜 덥게 느끼도록** 도와드릴 수 있어요.\n\n- 에어컨·선풍기로 실내 온도 낮추기  \n- 목, 손목, 겨드랑이에 시원한 물수건 대기  \n- 물이나 이온음료 조금씩 마시기  \n- 햇빛 강한 시간엔 커튼 치고 외출 줄이기  \n- “날씨가 나쁜 게 아니라 내가 지금 너무 더운 상태”라고 분리해서 생각하기\n\n그리고 마음속으로는 제가 주문 걸게요:  \n**“바람 선선하고, 습도 낮고, 구름 살짝 낀 완벽한 날씨로 변경!”** 🌿', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 185, 'prompt_tokens': 420, 'total_tokens': 605, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHIFFsvqZqHOcURgHm3sOebSvmEFI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0409c-e4ae-70b1-b282-41bb80

##### 세션(메모리) 방식의 문제점
- 메모리 저장이라 영속성이 없음
    - 서버 재시작/재배포 하면 store가 날아가서 히스토리도 같이 사라짐
- 세션 식별이 끊기기 쉬움
    - 쿠키/세션ID가 유지되지 않으면 같은 사람인지 매칭이 안 됨
- 스케일 아웃(서버 여러 대)에서 깨짐
    - A서버 메모리에 저장된 히스토리를 B서버는 모름 → 대화가 끊김

그래서 보통 이렇게 구성한다.
- SQLite/Redis/RDB 같은 저장소에 대화 내역을 저장해서
    - 사용자가 재접속해도 user_id 또는 thread_id로 복원
- 프롬프트에는 보통
    - 최근 N턴 + 요약 형태로 넣어서 비용/토큰도 관리